# Chapter 05: Supervised Learning
**Part II — Machine Learning Core**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

Linear/logistic regression, decision trees, random forests, gradient boosting, SVMs, and evaluation metrics.

## Learning Objectives
See Chapter 5 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Split data


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Split data
X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocess: scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Train a Random Forest
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_sc, y_train)

# Evaluate
y_pred = clf.predict(X_test_sc)
print(classification_report(y_test, y_pred))

## Train linear regression


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Train linear regression
lr = LinearRegression()
lr.fit(X_train, y_train)

# Evaluate
y_pred = lr.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:.3f}, Ru00b2: {r2:.3f}")

# Ridge regression (L2 regularisation)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

## Bias-variance tradeoff visualisation with polynomial regression


In [ ]:
# Bias-variance tradeoff visualisation with polynomial regression
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

np.random.seed(42)
n = 60
X = np.sort(np.random.uniform(0, 1, n))
y = np.sin(2 * np.pi * X) + np.random.normal(0, 0.3, n)
X_plot = np.linspace(0, 1, 200)

degrees = [1, 3, 10, 20]
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for ax, deg in zip(axes, degrees):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X.reshape(-1,1), y)
    y_pred = model.predict(X_plot.reshape(-1,1))
    train_score = model.score(X.reshape(-1,1), y)
    cv_score = cross_val_score(model, X.reshape(-1,1), y, cv=5).mean()
    
    ax.scatter(X, y, alpha=0.5, s=20, color='#1F3864')
    ax.plot(X_plot, np.sin(2*np.pi*X_plot), 'g--', lw=1.5, label='True fn')
    ax.plot(X_plot, y_pred, color='#C0392B', lw=2, label=f'Degree {deg}')
    ax.set_title(f'Degree {deg}\nTrain R²={train_score:.2f}, CV R²={cv_score:.2f}', fontsize=9)
    ax.set_ylim(-2.5, 2.5)
    ax.grid(alpha=0.3)
    if deg == 1:  ax.set_ylabel('y')

plt.suptitle('Bias-Variance Tradeoff: Polynomial Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch05_bias_variance.png', dpi=150, bbox_inches='tight')
plt.show()

## Precision-Recall vs ROC: imbalanced dataset


In [ ]:
# Precision-Recall vs ROC: imbalanced dataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                              average_precision_score, roc_auc_score)

# Imbalanced dataset: 5% positive
X, y = make_classification(n_samples=5000, n_features=20,
                            weights=[0.95, 0.05], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, proba)
prec, rec, _ = precision_recall_curve(y_test, proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, color='#2E75B6', lw=2,
             label=f'ROC (AUC = {roc_auc_score(y_test, proba):.3f})')
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec, prec, color='#C7770A', lw=2,
             label=f'PR (AP = {average_precision_score(y_test, proba):.3f})')
axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', lw=1, label='Baseline (random)')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Imbalanced Classification: ROC vs PR Curves (5% positive class)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ch05_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Accuracy of always-negative model: {1 - y_test.mean():.4f}")

---

## 📚 Review Questions

See Chapter 5 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 5 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*